## Overview

This is an implementation of the Levi–Perdew–Sahni density functional theory (LPS DFT) presented in [Levi.1984.PRA.30.2745]. The Kohn–Sham (KS) DFT code provided by Psi4 program is employed. Here, the most basic form of LPS DFT is implemented: spin-unrestricted formalism is applied (USCF), DIIS can be turned on/off by desire, the Core initial guess is used. The Pauli and exchange-correlation energy functionals are specified via the "custom DFT functional" Psi4 tool. 

The Fermi–Amaldi functional, which is the exact exchange of the LPS framework, can be calculated with a specified scaling factor.  

The dynamic damping is applied. It is dynamic in a sense that it takes a certain value before the specified threshold of a convegence criteria (`cutoff`) is reached and takes another value after. For example, the initial damping is set to `damp = 0.9` to avoid chaotic oscillations in the beginning of the SCF cycle. When a specified `cutoff` is reached, say the RMSD between an old and a new density matrices is below `1.0E-3`, the damping factor takes a value of `0.0`. 

In [1]:
import psi4
import numpy as np
from lps_uscf import lps_solver

## Tests

### LDA (exchange only) energy of H and He

The first test is to set $T_{\text{P}}[n] = 0$ and $E_{\text{xc}}[n] = E_{\text{x}}^{\text{LDA}}[n]$ and to reproduce the LDAx (Dirac) energy of the H and He atoms:

In [8]:
psi4.core.set_output_file('output.dat', False)

mol = psi4.geometry("""
units bohr
0 2
H
symmetry c1
""")
psi4.set_options({'basis': 'UGBS', 
                 'DFT_SPHERICAL_POINTS': 74,
                  'DFT_RADIAL_POINTS': 350})

TP = ['LDA_K_TF', 0.0]
LAMBDA = 1.0
EXC = ['LDA_X', 1.0, 'LDA_C_VWN', 0.0]
FA = [False, 1.0]
DIIS = True
MAX_ITER = 1000
DAMPING = [0.0, 0.0, 0.001]
D_guess = None
verbose=True

E, Da, Db, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,mol,DAMPING,FA,D_guess,DIIS,verbose)
print('\nFinal SCF energy: %.6f Hartree' % E)

Number of basis functions:   20

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:        -0.45553749   -4.55537E-01    2.06364E-03
SCF Iter  2:        -0.45705055   -1.51306E-03    3.55571E-04
SCF Iter  3:        -0.45707778   -2.72320E-05    4.54916E-05
SCF Iter  4:        -0.45707841   -6.22923E-07    2.81960E-06

Total time for SCF iterations: 0.251 seconds 

Final SCF energy: -0.457078 Hartree


In [ ]:
## The SCF type that matches our implementation is `PK`
psi4.core.set_output_file('output.dat', False)
psi4.set_options({'scf_type': 'PK',
                  'reference': 'UKS',
                  'basis': 'UGBS', 
                  'DFT_SPHERICAL_POINTS': 74,
                  'DFT_RADIAL_POINTS': 350})
XC = {
    "name": "XC",
    "x_functionals": {"LDA_X": {"alpha": 1.00}},
    "c_functionals": {"LDA_C_VWN": {"alpha": 0.00}}
}

ref_e = psi4.energy('SCF', dft_functional = XC)
print('Psi4 reference energy: %.6f Hartree' % ref_e)

Psi4 reference energy: -0.457078 Hartree


In [7]:
psi4.core.set_output_file('output.dat', False)

mol = psi4.geometry("""
units bohr
0 1
He
symmetry c1
""")
psi4.set_options({'basis': 'UGBS', 
                 'DFT_SPHERICAL_POINTS': 74,
                  'DFT_RADIAL_POINTS': 350})

TP = ['LDA_K_TF', 0.0]
LAMBDA = 1.0
EXC = ['LDA_X', 1.0, 'LDA_C_VWN', 0.0]
FA = [False, 1.0]
DIIS = True
MAX_ITER = 1000
DAMPING = [0.0, 0.0, 0.001]
D_guess = None
verbose=True

E, Da, Db, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,mol,DAMPING,FA,D_guess,DIIS,verbose)
print('\nFinal SCF energy: %.6f Hartree' % E)

Number of basis functions:   21

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:        -2.57214994   -2.57215E+00    5.71555E-02
SCF Iter  2:        -2.69124544   -1.19095E-01    2.50572E-02
SCF Iter  3:        -2.71709948   -2.58540E-02    1.19530E-02
SCF Iter  4:        -2.72314648   -6.04700E-03    3.25087E-03
SCF Iter  5:        -2.72363619   -4.89707E-04    2.73368E-04
SCF Iter  6:        -2.72363963   -3.43693E-06    8.79767E-06

Total time for SCF iterations: 0.369 seconds 

Final SCF energy: -2.723640 Hartree


In [6]:
psi4.core.set_output_file('output.dat', False)
psi4.set_options({'scf_type': 'PK',
                  'reference': 'UKS',
                  'basis': 'UGBS', 
                  'DFT_SPHERICAL_POINTS': 74,
                  'DFT_RADIAL_POINTS': 350})
XC = {
    "name": "XC",
    "x_functionals": {"LDA_X": {"alpha": 1.00}},
    "c_functionals": {"LDA_C_VWN": {"alpha": 0.00}}
}

ref_e = psi4.energy('SCF', dft_functional = XC)
print('Psi4 reference energy: %.6f Hartree' % ref_e)

Psi4 reference energy: -2.723640 Hartree


### Hartree–Fock energy for H and He

The second test is to set $T_{\text{P}}[n] = E_{\text{xc}}[n] = 0$, and to reproduce the HF energy of the He atom. Since, for one-orbital systems (like H or He atom) $E_{\text{x}}^{\text{HF}}[n] \equiv E_{\text{x}}^{\text{FA}}[n]$, calculating the HF energy of H and He in our implementation of LPS is equivalent to turning on the Fermi–Amaldi functional.

In [9]:
psi4.core.set_output_file('output.dat', False)

mol = psi4.geometry("""
units bohr
0 2
H
symmetry c1
""")
psi4.set_options({'basis': 'UGBS', 
                 'DFT_SPHERICAL_POINTS': 74,
                  'DFT_RADIAL_POINTS': 350})

TP = ['LDA_K_TF', 0.0]
LAMBDA = 1.0
EXC = ['LDA_X', 0.0, 'LDA_C_VWN', 0.0]
FA = [True, 1.0]
DIIS = True
MAX_ITER = 1000
DAMPING = [0.0, 0.0, 0.001]
D_guess = None
verbose=True

E, Da, Db, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,mol,DAMPING,FA,D_guess,DIIS,verbose)
print('\nFinal SCF energy: %.6f Hartree' % E)

Number of basis functions:   20

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:        -0.49999999   -5.00000E-01    7.61775E-13
SCF Iter  2:        -0.49999999   -1.11022E-16    8.01312E-13

Total time for SCF iterations: 0.120 seconds 

Final SCF energy: -0.500000 Hartree


In [10]:
## The SCF type that matches our implementation is `PK`
psi4.set_options({'scf_type': 'PK'})
ref_e = psi4.energy('SCF')
print('Psi4 reference energy: %.6f Hartree' % ref_e)

Psi4 reference energy: -0.500000 Hartree


In [11]:
psi4.core.set_output_file('output.dat', False)

mol = psi4.geometry("""
units bohr
0 1
He
symmetry c1
""")
psi4.set_options({'basis': 'UGBS', 
                 'DFT_SPHERICAL_POINTS': 74,
                  'DFT_RADIAL_POINTS': 350})

TP = ['LDA_K_TF', 0.0]
LAMBDA = 1.0
EXC = ['LDA_X', 0.0, 'LDA_C_VWN', 0.0]
FA = [True, 1.0]
DIIS = True
MAX_ITER = 1000
DAMPING = [0.0, 0.0, 0.001]
D_guess = None
verbose=True

E, Da, Db, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,mol,DAMPING,FA,D_guess,DIIS,verbose)
print('\nFinal SCF energy: %.6f Hartree' % E)

Number of basis functions:   21

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:        -2.74999994   -2.75000E+00    5.06114E-02
SCF Iter  2:        -2.85247100   -1.02471E-01    1.41793E-02
SCF Iter  3:        -2.86088368   -8.41268E-03    4.25445E-03
SCF Iter  4:        -2.86167286   -7.89181E-04    4.03652E-04
SCF Iter  5:        -2.86167982   -6.95934E-06    4.82780E-05
SCF Iter  6:        -2.86167993   -1.01062E-07    2.71332E-08

Total time for SCF iterations: 0.362 seconds 

Final SCF energy: -2.861680 Hartree


In [12]:
## The SCF type that matches our implementation is `PK`
psi4.set_options({'scf_type': 'PK'})
ref_e = psi4.energy('SCF')
print('Psi4 reference energy: %.6f Hartree' % ref_e)

Psi4 reference energy: -2.861680 Hartree


### Atomic energies from Table I in [Chan.2001.JCP.114.631]

The third test is to set $T_{\text{P}}[n] = T_{\text{TF}}[n]$ and $E_{\text{xc}}[n] = E^{\text{LDA}}_{\text{x}}[n]$, and to reproduce atomic energies reported in Table I of [Chan.2001.JCP.114.631] for different values of $\lambda$. 

In [15]:
psi4.core.set_output_file('output.dat', False)

mol = psi4.geometry("""
units bohr
0 1
He
symmetry c1
""")
psi4.set_options({'basis': 'Chan2001', 
                 'DFT_SPHERICAL_POINTS': 74,
                  'DFT_RADIAL_POINTS': 350})

TP = ['LDA_K_TF', 1.0]
LAMBDA = 1.0
EXC = ['LDA_X', 1.0, 'LDA_C_VWN', 0.0]
FA = [False, 1.0]
DIIS = True
MAX_ITER = 1000
DAMPING = [0.1, 0.0, 0.001]
D_guess = None
verbose=True

E, Da, Db, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,mol,DAMPING,FA,D_guess,DIIS,verbose)
print('\nFinal SCF energy: %.6f Hartree' % E)

Number of basis functions:   19

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:         1.09975190    1.09975E+00    4.35914E-01
SCF Iter  2:        -0.67263590   -1.77239E+00    1.97973E-02
SCF Iter  3:        -0.48608035    1.86556E-01    2.02675E-01
SCF Iter  4:        -0.60592505   -1.19845E-01    1.86198E-01
SCF Iter  5:        -0.69637800   -9.04530E-02    1.70851E-01
SCF Iter  6:        -1.41488956   -7.18512E-01    3.55964E-02
SCF Iter  7:        -1.44094951   -2.60599E-02    1.98657E-02
SCF Iter  8:        -1.09792140    3.43028E-01    1.19290E-01
SCF Iter  9:        -1.44943477   -3.51513E-01    2.37452E-02
SCF Iter 10:        -1.47520201   -2.57672E-02    1.12210E-03
SCF Iter 11:        -1.47720686   -2.00485E-03    4.38132E-04
SCF Iter 12:        -1.47744048   -2.33614E-04    6.04842E-05
SCF Iter 13:        -1.47744069   -2.16932E-07    1.95667E-05
SCF Iter 14:        -1.47744071   -1.47326E-08    9.50381E-07

Total time f

The SCF energy is $E_{\text{LPS}}[n] = -1.4774$ Hartree versus the reference value $E_{\text{Chan}}[n] = -1.4775$ Hartree. The chemical potential for He atom produced by our implementation is $\mu_{\text{LPS}} = -0.1082$ Hartree versus $\mu_{\text{Chan}} = -0.108$ Hartree.

## Reading an initial guess from another calculation

A density matrix from another SCF calculation can be employed as an initial guess. For example, calculate a TFDW energy and density of He to use it as an initial guess for the TFD0.2W calculation. 

In [24]:
psi4.core.clean_options()
psi4.core.clean()
psi4.core.set_output_file('output.dat', False)

mol = psi4.geometry("""
units bohr
0 1
He
symmetry c1
""")
psi4.set_options({'basis': 'Chan2001', ## Basis set described in [Chan.2001.JCP.114.631]
                 'DFT_SPHERICAL_POINTS': 6,   ## Minimal number of spherical points in Psi4
                  'DFT_RADIAL_POINTS': 1000}) ## Grid setting is close to the one described in [Chan.2001.JCP.114.631]

Pauli = {
    "name": "Pauli",
    "x_functionals": {"LDA_X": {"alpha": 0.00}},
    "c_functionals": {"LDA_K_TF": {"alpha": 1.00}} ## T_P = T_TF
}
XC = {
    "name": "XC",
    "x_functionals": {"LDA_X": {"alpha": 1.00}},  ## E_xc = E_LDAx
    "c_functionals": {"LDA_C_VWN": {"alpha": 0.00}}
}

## Damping factor before cutoff, after cutoff, the cutoff.
damp = [0.1, 0.0, 0.001]
## Calculate Fermi–Amaldi? Scaling factor.
FA = [False, 1.0]

SCF_E, D, SCF_ITER = lps_solver(1000, Pauli, XC, 1.0, mol, damp, FA, D_guess=None, DIIS=True)
print('\nFinal SCF energy: %.4f Hartree' % SCF_E)

Number of basis functions:   19

Starting SCF iterations:

    Iter               Energy         ChemPot       Delta E         dRMS

SCF Iter  1:         1.09975190   -1.99992E+00    1.09975E+00    4.35914E-01
SCF Iter  2:        -0.67262643    1.40907E-02   -1.77238E+00    1.97971E-02
SCF Iter  3:        -0.48607975   -1.22053E+00    1.86547E-01    2.02675E-01
SCF Iter  4:        -0.60591279   -9.99525E-01   -1.19833E-01    1.86200E-01
SCF Iter  5:        -0.69642376   -9.57175E-01   -9.05110E-02    1.70843E-01
SCF Iter  6:        -1.41488438   -2.83961E-01   -7.18461E-01    3.56035E-02
SCF Iter  7:        -1.44096627   -2.91318E-01   -2.60819E-02    1.98673E-02
SCF Iter  8:        -1.09782242   -4.40009E-01    3.43144E-01    1.19313E-01
SCF Iter  9:        -1.44947829   -1.45703E-01   -3.51656E-01    2.37240E-02
SCF Iter 10:        -1.47520218   -1.03484E-01   -2.57239E-02    1.12412E-03
SCF Iter 11:        -1.47720664   -1.09349E-01   -2.00446E-03    4.43514E-04
SCF Iter 12:        

In [28]:
damp = [0.9, 0.9, 0.001]

SCF_E, D, SCF_ITER = lps_solver(1000, Pauli, XC, 0.2, mol, damp, FA, D_guess=D, DIIS=True)
print('\nFinal SCF energy: %.4f Hartree' % SCF_E)

Number of basis functions:   19

Starting SCF iterations:

    Iter               Energy         ChemPot       Delta E         dRMS

SCF Iter  1:        -2.14035147    0.00000E+00   -2.14035E+00    1.93609E-01
SCF Iter  2:        -2.30880638   -1.30160E+00   -1.68455E-01    2.29005E-01
SCF Iter  3:        -2.42712437   -1.09064E+00   -1.18318E-01    2.40223E-01
SCF Iter  4:        -2.35845039    5.20194E-02    6.86740E-02    2.20884E-01
SCF Iter  5:        -2.28637693   -2.32298E-02    7.20735E-02    2.03649E-01
SCF Iter  6:        -2.19052253   -1.16702E-01    9.58544E-02    1.89457E-01
SCF Iter  7:        -2.26017058   -4.38461E+00   -6.96480E-02    2.21421E-01
SCF Iter  8:        -2.11576254   -5.49527E+00    1.44408E-01    2.13118E-01
SCF Iter  9:        -1.96295684   -3.76431E+00    1.52806E-01    2.24954E-01
SCF Iter 10:        -1.75599803   -3.58102E+00    2.06959E-01    2.56032E-01
SCF Iter 11:        -1.71711115   -1.41431E+00    3.88869E-02    2.73773E-01
SCF Iter 12:        